# Ho-Lee vs Hull-White FRA workflow
Calibrate both models on the same curve/option snapshot and compare FRA pricing, convexity and tail risk.

In [ ]:
import numpy as np
import pandas as pd
from src.models.short_rate import HoLeeModel, HullWhite1FModel, simulate_fra_distribution

valuation_date = '2026-04-05'
t = np.linspace(1/12, 5.0, 60)
curve = pd.DataFrame({'t': t, 'zero_rate': 0.04 - 0.002*np.exp(-t)})
market = pd.DataFrame({'expiry': [0.5, 1.0, 2.0, 3.0, 5.0], 'normal_vol': [0.010, 0.011, 0.012, 0.013, 0.0135]})

In [ ]:
ho = HoLeeModel()
hw = HullWhite1FModel()

ho.fit_initial_curve(curve)
hw.fit_initial_curve(curve)
print('Ho-Lee calibration:', ho.calibrate_to_options(market))
print('Hull-White calibration:', hw.calibrate_to_options(market))

In [ ]:
tenor = (0.5, 1.0)
fra_ho = simulate_fra_distribution(ho, curve, *tenor, n_paths=20_000, seed=7)
fra_hw = simulate_fra_distribution(hw, curve, *tenor, n_paths=20_000, seed=7)

summary = pd.DataFrame([
    {'model': 'Ho-Lee', 'pricing_error_proxy': fra_ho.pnl.mean(), 'convexity': fra_ho.futures_rate.mean()-fra_ho.fra_forward.mean(), 'tail_p01': np.quantile(fra_ho.pnl, 0.01)},
    {'model': 'Hull-White', 'pricing_error_proxy': fra_hw.pnl.mean(), 'convexity': fra_hw.futures_rate.mean()-fra_hw.fra_forward.mean(), 'tail_p01': np.quantile(fra_hw.pnl, 0.01)},
])
summary